In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load all 4 years
dfs = []
for year in [2019, 2022, 2023, 2024]:
    df = pd.read_csv(f'data/raw/PPR-{year}.csv', encoding='latin-1')
    df['year'] = year
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

# Rename columns
df.columns = ['date', 'address', 'county', 'eircode',
              'price', 'not_full_market', 'vat_exclusive',
              'description', 'size', 'year']

# Clean price
df['price'] = df['price'].str.replace(r'[^\d.]', '', regex=True).astype(float)

# Filter full market price only
df = df[df['not_full_market'] == 'No'].copy()

# Parse date
df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y')
df['month'] = df['date'].dt.month

# Property type simplified
df['property_type'] = df['description'].apply(
    lambda x: 'New' if 'New' in str(x) else 'Second-Hand'
)

print(f"Dataset loaded: {len(df):,} transactions")
print(f"Years: {sorted(df['year'].unique())}")
print(f"Counties: {df['county'].nunique()}")
print(f"\nMedian price by year:")
print(df.groupby('year')['price'].median().apply(lambda x: f'EUR {x:,.0f}'))

Dataset loaded: 232,317 transactions
Years: [np.int64(2019), np.int64(2022), np.int64(2023), np.int64(2024)]
Counties: 26

Median price by year:
year
2019    EUR 242,000
2022    EUR 290,000
2023    EUR 310,000
2024    EUR 335,000
Name: price, dtype: object
